In [5]:
%pip install torch torchvision matplotlib tqdm


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from torchvision.utils import save_image

# ========== Define Model Components ==========
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim=None):
        super().__init__()
        self.norm = nn.BatchNorm2d(in_ch)
        self.act = nn.SiLU()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_emb = nn.Linear(time_emb_dim, out_ch) if time_emb_dim else None

    def forward(self, x, t=None):
        h = self.act(self.norm(x))
        h = self.conv(h)
        if self.time_emb is not None and t is not None:
            h += self.time_emb(t)[:, :, None, None]
        return h

class Down(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.block1 = Block(in_ch, out_ch, time_emb_dim)
        self.block2 = Block(out_ch, out_ch, time_emb_dim)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x, t):
        h = self.block1(x, t)
        h = self.block2(h, t)
        return self.pool(h), h

class Up(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.block1 = Block(in_ch, out_ch, time_emb_dim)
        self.block2 = Block(out_ch, out_ch, time_emb_dim)

    def forward(self, x, skip, t):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        x = torch.cat([x, skip], dim=1)
        h = self.block1(x, t)
        h = self.block2(h, t)
        return h

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(1, dim)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim)

    def forward(self, t):
        t = t.float().unsqueeze(-1) / 1000
        t = self.act(self.fc1(t))
        t = self.fc2(t)
        return t

class StrongerUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, time_emb_dim=256):
        super().__init__()
        self.time_mlp = TimeEmbedding(time_emb_dim)
        self.down1 = Down(in_ch, 64, time_emb_dim)
        self.down2 = Down(64, 128, time_emb_dim)
        self.down3 = Down(128, 256, time_emb_dim)
        self.mid_block1 = Block(256, 256, time_emb_dim)
        self.mid_block2 = Block(256, 256, time_emb_dim)
        self.up3 = Up(512, 128, time_emb_dim)
        self.up2 = Up(256, 64, time_emb_dim)
        self.up1 = Up(128, 32, time_emb_dim)
        self.final_conv = nn.Conv2d(32, out_ch, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h1, skip1 = self.down1(x, t_emb)
        h2, skip2 = self.down2(h1, t_emb)
        h3, skip3 = self.down3(h2, t_emb)
        mid = self.mid_block1(h3, t_emb)
        mid = self.mid_block2(mid, t_emb)
        up3 = self.up3(mid, skip3, t_emb)
        up2 = self.up2(up3, skip2, t_emb)
        up1 = self.up1(up2, skip1, t_emb)
        return self.final_conv(up1)


In [2]:
def ddim_sample(model, x, T=300):
    model.eval()
    betas = torch.linspace(1e-4, 0.02, T).to(x.device)
    alphas = 1. - betas
    alpha_bars = torch.cumprod(alphas, dim=0)

    for t in reversed(range(T)):
        t_batch = torch.tensor([t], dtype=torch.long).to(x.device)
        eps_theta = model(x, t_batch)
        alpha_bar_t = alpha_bars[t]
        alpha_bar_t_sqrt = torch.sqrt(alpha_bar_t)
        one_minus_alpha_bar_sqrt = torch.sqrt(1 - alpha_bar_t)
        x0_pred = (x - one_minus_alpha_bar_sqrt * eps_theta) / alpha_bar_t_sqrt

        if t > 0:
            alpha_bar_prev = alpha_bars[t - 1]
            sigma = 0  # deterministic
            noise = torch.randn_like(x) if sigma > 0 else 0
            x = (torch.sqrt(alpha_bar_prev) * x0_pred +
                 torch.sqrt(1 - alpha_bar_prev - sigma**2) * eps_theta +
                 sigma * noise)
        else:
            x = x0_pred
    return x


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Step 1: Initialize model
model = StrongerUNet().to(device)

# Step 2: Load pretrained weights
checkpoint_path = r"C:/Users/bhoom/Downloads/ddim_epoch_50.pth"  # Adjust this if needed
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

# Step 3: Generate and save 5 images with different timestep T values
def generate_ddim_images(model, num_images=5, num_timesteps=50, save_dir="results4/final_samples", T=300):
    os.makedirs(save_dir, exist_ok=True)
    for i in range(1, num_images + 1):
        with torch.no_grad():
            noise = torch.randn(1, 3, 64, 64).to(device)
            sampled = ddim_sample(model, noise, T=T)
            sampled = (sampled + 1) / 2
            sampled = sampled.clamp(0, 1)
            save_path = os.path.join(save_dir, f"image_T{T}_{i}.png")
            save_image(sampled, save_path)
            print(f"Saved: {save_path}")

# Run with different T values (tuning)
generate_ddim_images(model, T=200)
generate_ddim_images(model, T=300)
generate_ddim_images(model, T=500)
generate_ddim_images(model, T=700)


Saved: results4/final_samples\image_T200_1.png
Saved: results4/final_samples\image_T200_2.png
Saved: results4/final_samples\image_T200_3.png
Saved: results4/final_samples\image_T200_4.png
Saved: results4/final_samples\image_T200_5.png
Saved: results4/final_samples\image_T300_1.png
Saved: results4/final_samples\image_T300_2.png
Saved: results4/final_samples\image_T300_3.png
Saved: results4/final_samples\image_T300_4.png
Saved: results4/final_samples\image_T300_5.png
Saved: results4/final_samples\image_T500_1.png
Saved: results4/final_samples\image_T500_2.png
Saved: results4/final_samples\image_T500_3.png
Saved: results4/final_samples\image_T500_4.png
Saved: results4/final_samples\image_T500_5.png
Saved: results4/final_samples\image_T700_1.png
Saved: results4/final_samples\image_T700_2.png
Saved: results4/final_samples\image_T700_3.png
Saved: results4/final_samples\image_T700_4.png
Saved: results4/final_samples\image_T700_5.png


In [9]:
%fidelity --gpu --isc --fid --input1 ./generated --input2 ./real


UsageError: Line magic function `%fidelity` not found.


In [20]:
import os
from PIL import Image, UnidentifiedImageError
from torchvision import transforms
import torch
from torchmetrics.image.inception import InceptionScore

device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),                             # [0, 1]
    transforms.Lambda(lambda x: (x * 255).byte())      # [0, 255] and uint8
])

imgs = []
folder_path = "results4/final_samples"  # ✅ change to your actual path
valid_extensions = ('.png', '.jpg', '.jpeg')

for filename in os.listdir(folder_path):
    if filename.lower().endswith(valid_extensions):
        file_path = os.path.join(folder_path, filename)
        try:
            img = Image.open(file_path).convert("RGB")
            imgs.append(transform(img))
        except UnidentifiedImageError:
            print(f"Skipped unreadable image: {file_path}")
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

if len(imgs) > 0:
    imgs_tensor = torch.stack(imgs).to(device)
    metric = InceptionScore().to(device)
    score = metric(imgs_tensor)
    print("✅ Inception Score:", score)
else:
    print("❌ No valid images found.")


✅ Inception Score: (tensor(1.5099), tensor(0.2526))
